# 1. Definition
Disjoint Set Union (DSU), also known as Union-Find, is a data structure that keeps track of a partition of a set into disjoint subsets. It provides efficient operations for merging subsets and finding the representative (or "parent") of a subset.

A DSU typically supports two main operations:
1. **Find**: This operation returns the representative (or parent) of the subset that a particular element belongs to. It is often implemented with path compression to optimize the time complexity.
2. **Union**: This operation merges two subsets into a single subset. It is often implemented with union by rank or union by size to keep the tree flat and optimize the time complexity.
3. **MakeSet**: This operation creates a new set containing a single element.


In [1]:
class DisjointSetUnion:
    def __init__(self, n):
        """Initialize the DSU with n elements."""
        self._parents = list(range(n))
        self._rank = [0] * n  # Used for union by rank optimization


    def find(self, x):
        """Find the representative(root) of the set that contains x."""
        if self._parents[x] != x:
            self._parents[x] = self.find(self._parents[x])  # Path compression
        return self._parents[x]

    def union(self, x, y):
        """Merge the sets that contain x and y."""
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            # Union by rank optimization
            if self._rank[root_x] > self._rank[root_y]:
                self._parents[root_y] = root_x
            elif self._rank[root_x] < self._rank[root_y]:
                self._parents[root_x] = root_y
            else:
                self._parents[root_y] = root_x
                self._rank[root_x] += 1
    
    def make_set(self, x):
        """Create a new set containing the single element x."""
        if x >= len(self._parents):
            self._parents.extend(range(len(self._parents), x + 1))  # Extend the parents list if x is out of bounds
            self._rank.extend([0] * (x + 1 - len(self._rank)))  # Extend the rank list accordingly
        else:
            raise ValueError("Element already exists in the DSU.")

# Example usage:
dsu = DisjointSetUnion(5)  # Initialize DSU with 5 elements
dsu.union(0, 1)  # Merge sets containing 0 and 1
dsu.union(2, 3)  # Merge sets containing 2 and 3
dsu.union(1, 2)  # Merge sets containing 1 and 2 (which also merges 0, 1, 2, and 3)
dsu.make_set(5)  # Create a new set containing the single element 5
print(dsu.find(0))  # Output: 0 (or 1, 2, or 3, depending on the union operations)
print(dsu.find(3))  # Output: 4 (since it's a new set)
print(dsu.find(5))  # Output: 5 (since it's a new set)

0
0
5


# 2. Ackerman's function and Inverse Ackerman's function
> Which will be used in the analysis of the time complexity of DSU.
Ackerman's function is a recursive function that grows very rapidly. It is defined as follows:
$$A(m, n) = \begin{cases}
n + 1 & \text{if } m = 0 \\
A(m - 1, 1) & \text{if } m > 0 \text{ and } n = 0 \\
A(m - 1, A(m, n - 1)) & \text{if } m > 0 \text{ and }n > 0
\end{cases}$$

And the **inverse Ackermann function** is defined as the function that grows extremely slowly, essentially the inverse of the Ackermann function in terms of its growth rate. It is often denoted as $\alpha(n)$ and is used in the analysis of the disjoint set union (DSU) data structure. The inverse Ackermann function can be defined as follows:
$$\alpha(n) = \min \{ m \geq 0 : A(m, n) \geq n \}$$

# 3. Time Complexity Analysis of DSU

## 3.1 properties of DSU
The DSU we implemented has the following properties:
1. $\forall x \in S$, we have $rank(x) < rank(parent(x))$
   1. That's because when we merge two sets, we always attach the smaller tree to the larger tree, which ensures that the rank of the parent is always greater than the rank of the child.
2. Each node $x$ with rank $k$, its subtree has at least $2^k$ nodes.
   1. This can be proved by induction. 
      1. Base case: when $k = 0$, the subtree has at least $2^0 = 1$ node, which is true.
      2. Induction step: Each node with rank $k$ is formed by merging two trees of rank $k-1$, which means that the subtree rooted at this node has at least $2^{k-1} + 2^{k-1} = 2^k$ nodes.
3. The maximum rank of any node is at most $\log n$.

By the following properties, we can analysis the time complexity by **Accounting Method**.

## 3.2 Level of a node
First of all, we define the **level** of a node $x$. For each node $x$, we define the following function:
$$
level(x) = \text{the largest integer } k,s.t. A(k, rank(x)) \leq rank(parent(x))
$$
The level of a node measures how many times that the rank of the parent is greater than the rank of the node itself. 
> For example, if $level(x) = 0$, then $rank(parent(x)) < A(1, rank(x)) = rank(x) + 1$, which means that the rank of the parent is at most the rank of the node itself. If $level(x) = 1$, then $rank(parent(x)) < A(2, rank(x)) = 2^{rank(x)}$, which means that the rank of the parent is at most $2^{rank(x)}$. If $level(x) = 2$, then $rank(parent(x)) < A(3, rank(x)) = 2^{2^{rank(x)}}$, which means that the rank of the parent is at most $2^{2^{rank(x)}}$. And so on.

Because of the properties 2, we have:
$$
A(\alpha(n), rank(x)) \geq \log_2{n}\geq rank(parent(x))
$$
which means that $level(x) \leq \alpha(n)$ for all nodes $x$.

......
